# Orion-XAI — QLoRA Fine-Tuning Notebook
**Model:** Llama-3.2-3B-Instruct  
**Method:** QLoRA (4-bit NF4) via Unsloth  
**Dataset:** xai_cybersec_1000.jsonl (1000 XAI/cybersecurity Alpaca examples)  
**Hardware:** Kaggle T4 GPU (16 GB VRAM)  

> After training, download the saved adapter from `/kaggle/working/orion-xai-adapter/`

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install unsloth trl peft bitsandbytes accelerate datasets transformers --upgrade -q

## Step 2 — Load base model with 4-bit quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None           # Auto-detect (float16 on T4)
LOAD_IN_4BIT = True    # QLoRA 4-bit NF4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name  = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

print("Base model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Step 3 — Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,          # LoRA rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha     = 32,          # alpha = 2 * r
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth optimised checkpointing
    random_state   = 42,
    use_rslora     = False,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 4 — Load and format the dataset

In [ ]:
from datasets import load_dataset

# ----- Locate dataset file -----
# Option A: uploaded as a Kaggle Dataset (recommended)
#   Path: /kaggle/input/<your-dataset-name>/xai_cybersec_1000.jsonl
# Option B: uploaded directly to this notebook session
#   Path: /kaggle/working/xai_cybersec_1000.jsonl

import os, glob
candidates = glob.glob("/kaggle/input/**/xai_cybersec_1000.jsonl", recursive=True)
if candidates:
    DATASET_PATH = candidates[0]
else:
    DATASET_PATH = "/kaggle/working/xai_cybersec_1000.jsonl"

print(f"Using dataset: {DATASET_PATH}")
assert os.path.exists(DATASET_PATH), f"Dataset not found at {DATASET_PATH}"

# Load raw JSONL
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Examples loaded: {len(raw_dataset)}")
print("Sample:", raw_dataset[0])

In [ ]:
# Alpaca prompt template
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token  # <|eot_id|> for Llama 3

def format_example(examples):
    texts = []
    for instr, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        text = ALPACA_PROMPT.format(
            instruction=instr,
            input=inp if inp else "",
            output=out,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.map(format_example, batched=True)

# Quick sanity check — print one formatted example
print(dataset[0]["text"][:500])

## Step 5 — Configure training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing          = True,   # Pack short examples together — improves GPU util
    args = TrainingArguments(
        per_device_train_batch_size   = 2,
        gradient_accumulation_steps   = 4,   # Effective batch = 8
        warmup_ratio                  = 0.05,
        num_train_epochs              = 3,
        learning_rate                 = 2e-4,
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        lr_scheduler_type             = "cosine",
        seed                          = 42,
        output_dir                    = "/kaggle/working/orion-xai-checkpoints",
        save_strategy                 = "epoch",
        report_to                     = "none",
    ),
)

print("Trainer configured.")

## Step 6 — Train

In [ ]:
# Show GPU stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Reserved: {start_gpu_memory} GB")

# Train
trainer_stats = trainer.train()

# Show GPU stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"\nTraining complete!")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Train loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 7 — Test the model before saving

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

test_prompt = ALPACA_PROMPT.format(
    instruction="Explain what SHAP values are and why they are used in cybersecurity ML models.",
    input="",
    output="",  # Empty — model fills this in
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens   = 400,
    temperature      = 0.7,
    do_sample        = True,
    pad_token_id     = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("ORION-XAI RESPONSE:")
print("=" * 60)
print(response.strip())

## Step 8 — Save LoRA adapter

In [ ]:
ADAPTER_PATH = "/kaggle/working/orion-xai-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"Adapter saved to: {ADAPTER_PATH}")
print("\nFiles saved:")
import os
for f in os.listdir(ADAPTER_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

## Step 9 — (Optional) Merge + save full model

Only run this if you want to save the merged model directly from Kaggle.
The merged model is ~6 GB — may hit Kaggle's 20 GB output limit.
**Recommended:** download the adapter (~50 MB) and merge locally.


In [ ]:
# OPTIONAL — run only if you want merged model on Kaggle
# Uncomment to use:

# MERGED_PATH = "/kaggle/working/orion-xai-merged"
# model.save_pretrained_merged(
#     MERGED_PATH,
#     tokenizer,
#     save_method = "merged_16bit",
# )
# print(f"Merged model saved to: {MERGED_PATH}")

print("Skipping merge — download adapter and merge locally.")
print("Download: /kaggle/working/orion-xai-adapter/")

## Done! ✅

**Next steps after downloading the adapter:**

```bash
# 1. Merge adapter with base model (run on your Windows machine)
python merge_and_convert.py

# 2. Convert merged model to GGUF
python llama.cpp/convert_hf_to_gguf.py orion-xai-merged --outtype f16 --outfile orion-xai-f16.gguf

# 3. Reload into Ollama
ollama rm orion-xai
ollama create orion-xai -f Modelfile

# 4. Test
ollama run orion-xai "Explain SHAP values for IDS detection"
```